In [1]:
# Register the Gold CSVs as SQL views so queries can use plain table names
# (fact_order_items, dim_products, ...), exactly as in the final .sql files.
import os
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DecimalType

spark = (SparkSession.builder.master("local[*]").appName("blue-owls-sql")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

GOLD_DIR = "/home/jovyan/work/output/gold"
MONEY = DecimalType(12, 2)

def load(name):
    return spark.read.option("header", True).csv(f"{GOLD_DIR}/{name}.csv")

# CSV columns arrive as text, so restore the types the queries need.
fact = (load("fact_order_items")
        .withColumn("price", F.col("price").cast(MONEY))
        .withColumn("freight_value", F.col("freight_value").cast(MONEY))
        .withColumn("payment_value", F.col("payment_value").cast(MONEY))
        .withColumn("days_delivery_vs_estimate", F.col("days_delivery_vs_estimate").cast("int"))
        .withColumn("is_late_delivery", F.col("is_late_delivery").cast("boolean"))
        .withColumn("order_date", F.to_date("order_date")))

prod = load("dim_products")

fact.createOrReplaceTempView("fact_order_items")
prod.createOrReplaceTempView("dim_products")
load("dim_customers").createOrReplaceTempView("dim_customers")
load("dim_sellers").createOrReplaceTempView("dim_sellers")

print("Views ready:", [t.name for t in spark.catalog.listTables()])

Views ready: ['dim_customers', 'dim_products', 'dim_sellers', 'fact_order_items']


In [2]:
os.makedirs("/home/jovyan/work/sql", exist_ok=True)
print("sql folder ready")

sql folder ready


In [3]:
%%writefile /home/jovyan/work/sql/query_1.sql
-- Query 1: Revenue trend analysis with ranking (Spark SQL)
--
-- Approach:
--   1. monthly : revenue (price + freight_value) and transaction count per category
--                per month. HAVING removes category-months with fewer than 10 transactions.
--   2. ranked  : rank every qualifying category inside its month by revenue.
--                The rank is computed over ALL categories, before narrowing to the top 5,
--                so monthly_rank means "rank among all categories that month".
--                RANK() is used, so tied revenues share the same rank.
--   3. top5    : the 5 categories with the highest overall revenue across the whole period.
--   4. trend   : keep only the top 5 and add the previous month's revenue (LAG) and a
--                3-month rolling average. month_idx = year * 12 + month is a consecutive
--                number per calendar month, so "previous month" and "last 3 months" are
--                real calendar months even when a month is missing for a category.
--   5. final   : mom_growth_pct is NULL when the immediately preceding calendar month
--                has no row for the category (first month, or a gap), because a growth
--                figure against a non-adjacent month would be misleading.

WITH monthly AS (
    SELECT
        p.product_category_name,
        YEAR(f.order_date)                                AS year,
        MONTH(f.order_date)                               AS month,
        YEAR(f.order_date) * 12 + MONTH(f.order_date)     AS month_idx,
        SUM(f.price + f.freight_value)                    AS monthly_revenue,
        COUNT(*)                                          AS transactions
    FROM fact_order_items f
    JOIN dim_products p ON f.product_key = p.product_key
    GROUP BY p.product_category_name, YEAR(f.order_date), MONTH(f.order_date)
    HAVING COUNT(*) >= 10
),

ranked AS (
    SELECT
        monthly.*,
        RANK() OVER (PARTITION BY month_idx ORDER BY monthly_revenue DESC) AS monthly_rank
    FROM monthly
),

top5 AS (
    SELECT p.product_category_name
    FROM fact_order_items f
    JOIN dim_products p ON f.product_key = p.product_key
    GROUP BY p.product_category_name
    ORDER BY SUM(f.price + f.freight_value) DESC
    LIMIT 5
),

trend AS (
    SELECT
        product_category_name,
        year,
        month,
        month_idx,
        monthly_revenue,
        monthly_rank,
        LAG(monthly_revenue) OVER (PARTITION BY product_category_name ORDER BY month_idx) AS prev_revenue,
        LAG(month_idx)       OVER (PARTITION BY product_category_name ORDER BY month_idx) AS prev_month_idx,
        AVG(monthly_revenue) OVER (PARTITION BY product_category_name
                                   ORDER BY month_idx
                                   RANGE BETWEEN 2 PRECEDING AND CURRENT ROW) AS rolling_avg
    FROM ranked
    WHERE product_category_name IN (SELECT product_category_name FROM top5)
)

SELECT
    product_category_name,
    year,
    month,
    monthly_revenue,
    monthly_rank,
    CASE WHEN prev_month_idx = month_idx - 1 AND prev_revenue > 0
         THEN ROUND((monthly_revenue - prev_revenue) * 100.0 / prev_revenue, 2)
    END                          AS mom_growth_pct,
    ROUND(rolling_avg, 2)        AS rolling_3m_avg_revenue
FROM trend
ORDER BY product_category_name, year, month

Writing /home/jovyan/work/sql/query_1.sql


In [4]:
# Run the query from the saved file, so the file itself is what gets tested.
with open("/home/jovyan/work/sql/query_1.sql") as f:
    q1 = f.read().strip().rstrip(";")

result = spark.sql(q1)
result.show(60, truncate=False)

+---------------------+----+-----+---------------+------------+--------------+----------------------+
|product_category_name|year|month|monthly_revenue|monthly_rank|mom_growth_pct|rolling_3m_avg_revenue|
+---------------------+----+-----+---------------+------------+--------------+----------------------+
|beleza_saude         |2018|7    |122743.98      |1           |NULL          |122743.98             |
|beleza_saude         |2018|8    |137206.07      |1           |11.78         |129975.03             |
|cama_mesa_banho      |2018|7    |68708.38       |4           |NULL          |68708.38              |
|cama_mesa_banho      |2018|8    |75207.83       |3           |9.46          |71958.11              |
|esporte_lazer        |2018|7    |65830.02       |5           |NULL          |65830.02              |
|esporte_lazer        |2018|8    |60374.75       |5           |-8.29         |63102.39              |
|relogios_presentes   |2018|7    |106064.72      |2           |NULL          |1060

In [5]:
# Independent check: recompute the same top-5 / qualifying-months revenue with the
# DataFrame API and compare the total with the SQL result.
rev = F.col("price") + F.col("freight_value")
joined = fact.join(prod, "product_key")

top5 = [r[0] for r in (joined.groupBy("product_category_name")
                       .agg(F.sum(rev).alias("rev")).orderBy(F.desc("rev")).limit(5).collect())]
print("Top 5 categories:", top5)

expected = (joined.filter(F.col("product_category_name").isin(top5))
            .groupBy("product_category_name", F.year("order_date").alias("y"), F.month("order_date").alias("m"))
            .agg(F.sum(rev).alias("rev"), F.count("*").alias("n"))
            .filter("n >= 10")
            .agg(F.sum("rev")).first()[0])
actual = result.agg(F.sum("monthly_revenue")).first()[0]

print("expected:", expected, " actual:", actual, " ->", "PASS" if expected == actual else "FAIL")
print("distinct categories in result:", result.select("product_category_name").distinct().count(), "(should be 5)")
print("first-month growth values are NULL:",
      result.filter(F.col("mom_growth_pct").isNull()).count(), "row(s) with NULL growth")

Top 5 categories: ['beleza_saude', 'relogios_presentes', 'utilidades_domesticas', 'cama_mesa_banho', 'esporte_lazer']
expected: 867734.23  actual: 867734.23  -> PASS
distinct categories in result: 5 (should be 5)
first-month growth values are NULL: 5 row(s) with NULL growth
